[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonasneves/colab-slm-playground/blob/main/trending_gguf_chatbot_colab.ipynb)

# Trending GGUF Chatbot

Fetches trending GGUF models from Hugging Face, lets you pick a model and quantization variant, and runs a chat UI inside Colab.

Uses [`llama-cpp-python`](https://github.com/abetlen/llama-cpp-python) — runs on **CPU**, no GPU required.
Smaller quants (Q4_K_M) are faster; larger quants (Q8_0) are higher quality.

## 1 — Install dependencies

Builds `llama-cpp-python` with OpenBLAS for CPU acceleration. Takes ~2 minutes.

In [ ]:
import os
os.environ["CMAKE_ARGS"] = "-DGGML_BLAS=ON -DGGML_BLAS_VENDOR=OpenBLAS"
os.environ["FORCE_CMAKE"] = "1"

!pip install -q ninja llama-cpp-python huggingface_hub
!curl -sL https://raw.githubusercontent.com/jonasneves/colab-slm-playground/main/chat_ui.py -o /content/chat_ui.py

## 2 — Fetch trending models

In [ ]:
from huggingface_hub import HfApi
from chat_ui import build_model_table_html, register_load_callback
from IPython.display import display, HTML
import os
from llama_cpp import Llama

llm = None
loaded_model_id = None
loaded_variant = None

print("Fetching trending GGUF text-generation models...")

api = HfApi()
raw_models = list(api.list_models(
    filter="gguf",
    pipeline_tag="text-generation",
    sort="trending_score",
    limit=30,
    full=True,
))

_QUANT_RANK = ["Q4_K_M", "Q4_K_S", "Q4_0", "Q5_K_M", "Q5_K_S", "Q3_K_M", "Q8_0", "Q2_K"]

def _sort_gguf_files(files):
    def rank(f):
        upper = f.upper()
        for i, q in enumerate(_QUANT_RANK):
            if q in upper:
                return i
        return len(_QUANT_RANK)
    return sorted(files, key=rank)

model_info = []
for m in raw_models:
    gguf_files = [
        s.rfilename for s in (m.siblings or [])
        if s.rfilename.endswith(".gguf")
    ]
    if not gguf_files:
        continue
    model_info.append({
        "id": m.id,
        "downloads": m.downloads or 0,
        "likes": m.likes or 0,
        "gguf_files": _sort_gguf_files(gguf_files),
    })

def load_model(model_id, variant=""):
    global llm, loaded_model_id, loaded_variant
    llm = Llama.from_pretrained(
        repo_id=model_id,
        filename=variant,
        n_ctx=4096,
        n_threads=os.cpu_count(),
        verbose=False,
    )
    loaded_model_id = model_id
    loaded_variant = variant

register_load_callback(load_model)
display(HTML(build_model_table_html(model_info)))
print(f"Found {len(model_info)} models. Pick a variant and click Load.")

## 3 — Launch Chat UI

Click **Load** in the table above (pick a quant variant first). Once loaded, run this cell.

In [ ]:
from chat_ui import build_chat_html, register_callback
from IPython.display import display, HTML

class _Stop(Exception):
    def _render_traceback_(self): return []

if llm is None:
    print("Load a model first — pick a variant and click Load in the table above.")
    raise _Stop()

def generate(messages: list) -> str:
    response = llm.create_chat_completion(
        messages=messages,
        max_tokens=512,
        temperature=0.7,
        top_k=50,
        repeat_penalty=1.05,
    )
    return response["choices"][0]["message"]["content"].strip()

register_callback(generate)
short_name = loaded_model_id.split("/")[-1]
quant_label = loaded_variant.split("/")[-1] if loaded_variant else "GGUF"
display(HTML(build_chat_html(short_name, f"GGUF &middot; {quant_label}")))
print("Chat ready. Response time depends on model size and quant — expect 5–30 s on CPU.")

---
### Notes

**What is GGUF?**
GGUF (GPT-Generated Unified Format) is the file format used by [llama.cpp](https://github.com/ggerganov/llama.cpp). It packages weights, tokenizer, and model metadata into a single self-contained file, with built-in support for quantization at multiple precision levels. This makes it the most portable format for running models locally or on CPU — no Python ML framework required at the C++ level.

**What do the quant levels mean?**
The naming follows a pattern: `Q` = quantized, the number is bits per weight, `K` = k-quants (a more accurate quantization method), `M/S/L` = medium/small/large variant within that bit width.
- **Q2_K** — 2-bit, smallest file, noticeable quality loss
- **Q4_K_M** — 4-bit medium, best speed/quality tradeoff for most use cases
- **Q5_K_M** — 5-bit medium, closer to full quality, ~25% larger than Q4
- **Q8_0** — 8-bit, near-lossless, roughly 2× the size of Q4

The file dropdown defaults to Q4_K_M when available.

**Why CPU for GGUF?**
llama.cpp was built for CPU-first inference with BLAS acceleration (OpenBLAS, BLIS, Apple Accelerate). This notebook compiles with OpenBLAS, which vectorizes matrix multiplications across available cores. GPU support exists but requires a separate CUDA build; the other notebooks cover the GPU case via PyTorch.

**Context length (`n_ctx`)**
This sets the maximum number of tokens the model holds in its attention window — both the prompt and the generated response count toward it. 4096 is a safe default. Larger values increase RAM usage linearly; reduce it if you load a large model and hit memory limits.